In [1]:
# importing libraries
import pandas as pd
import numpy as np

In [2]:
# Load Movies Dataset
i_cols = ['movie_id', 'movie_title', 'release_date', 'video_release_date', 'IMDb_URL',
          'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy',
          'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
          'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

items = pd.read_csv('ml-100k/u.item', sep='|', names=i_cols, encoding='latin-1')
items.head(3)

,movie_id,movie_title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [3]:
# Load Ratings Dataset
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=r_cols, encoding='latin-1')
ratings.head(3)

,user_id,movie_id,rating,unix_timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116


# Step 2: Calculate Popularity Score
# Popularity = Rating Count + Average Rating combined using Weighted Score

In [6]:
# Calculate rating count and average rating for each movie
movie_stats = ratings.groupby('movie_id').agg(
    rating_count=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()
# Minimum votes threshold (at least 10 ratings to be considered popular)
m = 10
C = movie_stats['avg_rating'].mean()  # Global average rating

print(f"Global Average Rating (C): {C:.2f}")
print(f"Minimum votes threshold (m): {m}")


Global Average Rating (C): 3.08
Minimum votes threshold (m): 10


In [8]:
# Weighted Rating Formula (IMDB style)
# WR = (v / (v + m)) * R + (m / (v + m)) * C
# v = number of votes, m = min votes, R = avg rating, C = global avg

def weighted_rating(x, m=m, C=C):
    v = x['rating_count']
    R = x['avg_rating']
    return (v / (v + m)) * R + (m / (v + m)) * C

movie_stats['popularity_score'] = movie_stats.apply(weighted_rating, axis=1)

movie_stats.head()

,movie_id,rating_count,avg_rating,popularity_score
0,1,452,3.878319,3.860953
1,2,131,3.206107,3.196883
2,3,90,3.033333,3.037604
3,4,209,3.550239,3.528587
4,5,86,3.302326,3.278755


# Step 3: Merge Movie Info with Popularity Score

In [9]:
# Genre columns list
genres = ['Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime',
          'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical',
          'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

# Select needed columns from items
movie_content = items[['movie_id', 'movie_title'] + genres]

# Merge with popularity stats
movie_full = pd.merge(movie_content, movie_stats, on='movie_id', how='inner')

print(f"Total movies with ratings: {len(movie_full)}")
movie_full.head(3)

Total movies with ratings: 1682


,movie_id,movie_title,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,...,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,rating_count,avg_rating,popularity_score
0,1,Toy Story (1995),0,0,1,1,1,0,0,0,...,0,0,0,0,0,0,0,452,3.878319,3.860953
1,2,GoldenEye (1995),1,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,131,3.206107,3.196883
2,3,Four Rooms (1995),0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,90,3.033333,3.037604


# Step 4: Genre-wise Top 5 Popular Movies Function

In [11]:
def get_top5_by_genre(genre, df=movie_full, top_n=5):
   
    if genre not in genres:
        print(f"❌ Genre '{genre}' not found! Available genres: {genres}")
        return None
    
    # Filter movies belonging to the given genre (genre column = 1)
    genre_movies = df[df[genre] == 1].copy()
    
    # Sort by popularity score (descending)
    top_movies = genre_movies.sort_values('popularity_score', ascending=False).head(top_n)
    
    # Select display columns
    result = top_movies[['movie_id', 'movie_title', 'rating_count', 'avg_rating', 'popularity_score']].reset_index(drop=True)
    result.index = result.index + 1  # Start rank from 1
    result.index.name = 'Rank'
    result.columns = ['Movie ID', 'Movie Title', 'No. of Ratings', 'Avg Rating', 'Popularity Score']
    result['Avg Rating'] = result['Avg Rating'].round(2)
    result['Popularity Score'] = result['Popularity Score'].round(4)
    
    return result

In [12]:
# =============================
# ACTION - Top 5 Popular Movies
# =============================
print("🎬 TOP 5 POPULAR MOVIES - ACTION")
print("=" * 55)
display(get_top5_by_genre('Action'))

🎬 TOP 5 POPULAR MOVIES - ACTION


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,50,Star Wars (1977),583,4.36,4.3369
2,127,"Godfather, The (1972)",413,4.28,4.2548
3,174,Raiders of the Lost Ark (1981),420,4.25,4.2250
4,313,Titanic (1997),350,4.25,4.2132
5,172,"Empire Strikes Back, The (1980)",367,4.20,4.1744


In [13]:
# ================================
# ADVENTURE - Top 5 Popular Movies
# ================================
print("🎬 TOP 5 POPULAR MOVIES - ADVENTURE")
print("=" * 55)
display(get_top5_by_genre('Adventure'))

🎬 TOP 5 POPULAR MOVIES - ADVENTURE


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,50,Star Wars (1977),583,4.36,4.3369
2,174,Raiders of the Lost Ark (1981),420,4.25,4.2250
3,172,"Empire Strikes Back, The (1980)",367,4.20,4.1744
4,511,Lawrence of Arabia (1962),173,4.23,4.1681
5,173,"Princess Bride, The (1987)",324,4.17,4.1400


In [14]:
# =================================
# ANIMATION - Top 5 Popular Movies
# =================================
print(" TOP 5 POPULAR MOVIES - ANIMATION")
print("=" * 55)
display(get_top5_by_genre('Animation'))

 TOP 5 POPULAR MOVIES - ANIMATION


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,408,"Close Shave, A (1995)",112,4.49,4.3751
2,169,"Wrong Trousers, The (1993)",118,4.47,4.3575
3,114,Wallace & Gromit: The Best of Aardman Animatio...,67,4.45,4.2696
4,189,"Grand Day Out, A (1992)",66,4.11,3.9705
5,1,Toy Story (1995),452,3.88,3.8610


In [15]:
# ==================================
# CHILDREN'S - Top 5 Popular Movies
# ==================================
print(" TOP 5 POPULAR MOVIES - CHILDREN'S")
print("=" * 55)
display(get_top5_by_genre("Children's"))

 TOP 5 POPULAR MOVIES - CHILDREN'S


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,132,"Wizard of Oz, The (1939)",246,4.08,4.0381
2,8,Babe (1995),219,4.00,3.9553
3,1,Toy Story (1995),452,3.88,3.8610
4,423,E.T. the Extra-Terrestrial (1982),300,3.83,3.8089
5,95,Aladdin (1992),219,3.81,3.7806


In [16]:
# ==============================
# COMEDY - Top 5 Popular Movies
# ==============================
print(" TOP 5 POPULAR MOVIES - COMEDY")
print("=" * 55)
display(get_top5_by_genre('Comedy'))

 TOP 5 POPULAR MOVIES - COMEDY


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,408,"Close Shave, A (1995)",112,4.49,4.3751
2,169,"Wrong Trousers, The (1993)",118,4.47,4.3575
3,480,North by Northwest (1959),179,4.28,4.2210
4,173,"Princess Bride, The (1987)",324,4.17,4.1400
5,316,As Good As It Gets (1997),112,4.20,4.1046


In [17]:
# =============================
# CRIME - Top 5 Popular Movies
# =============================
print("TOP 5 POPULAR MOVIES - CRIME")
print("=" * 55)
display(get_top5_by_genre('Crime'))

TOP 5 POPULAR MOVIES - CRIME


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,12,"Usual Suspects, The (1995)",267,4.39,4.3385
2,127,"Godfather, The (1972)",413,4.28,4.2548
3,187,"Godfather: Part II, The (1974)",209,4.19,4.1359
4,100,Fargo (1996),508,4.16,4.1347
5,302,L.A. Confidential (1997),297,4.16,4.1263


In [18]:
# ==================================
# DOCUMENTARY - Top 5 Popular Movies
# ==================================
print(" TOP 5 POPULAR MOVIES - DOCUMENTARY")
print("=" * 55)
display(get_top5_by_genre('Documentary'))

 TOP 5 POPULAR MOVIES - DOCUMENTARY


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,48,Hoop Dreams (1994),117,4.09,4.0139
2,1142,When We Were Kings (1996),44,4.05,3.8659
3,813,"Celluloid Closet, The (1995)",56,3.89,3.7691
4,320,Paradise Lost: The Child Murders at Robin Hood...,20,4.05,3.7253
5,32,Crumb (1994),81,3.79,3.7117


In [19]:
# =============================
# DRAMA - Top 5 Popular Movies
# =============================
print("TOP 5 POPULAR MOVIES - DRAMA")
print("=" * 55)
display(get_top5_by_genre('Drama'))

TOP 5 POPULAR MOVIES - DRAMA


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,318,Schindler's List (1993),298,4.47,4.4213
2,483,Casablanca (1942),243,4.46,4.4022
3,64,"Shawshank Redemption, The (1994)",283,4.45,4.3985
4,98,"Silence of the Lambs, The (1991)",390,4.29,4.2594
5,127,"Godfather, The (1972)",413,4.28,4.2548


In [20]:
# =================================
# FANTASY - Top 5 Popular Movies
# =================================
print("TOP 5 POPULAR MOVIES - FANTASY")
print("=" * 55)
display(get_top5_by_genre('Fantasy'))

TOP 5 POPULAR MOVIES - FANTASY


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,423,E.T. the Extra-Terrestrial (1982),300,3.83,3.8089
2,558,Heavenly Creatures (1994),70,3.67,3.5970
3,1293,Star Kid (1997),3,5.00,3.5200
4,141,"20,000 Leagues Under the Sea (1954)",72,3.50,3.4483
5,755,Jumanji (1995),96,3.31,3.2902


In [21]:
# ==================================
# FILM-NOIR - Top 5 Popular Movies
# ==================================
print(" TOP 5 POPULAR MOVIES - FILM-NOIR")
print("=" * 55)
display(get_top5_by_genre('Film-Noir'))

 TOP 5 POPULAR MOVIES - FILM-NOIR


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,657,"Manchurian Candidate, The (1962)",131,4.26,4.1756
2,484,"Maltese Falcon, The (1941)",138,4.21,4.1335
3,302,L.A. Confidential (1997),297,4.16,4.1263
4,89,Blade Runner (1982),275,4.14,4.1009
5,654,Chinatown (1974),147,4.14,4.0685


In [22]:
# ==============================
# HORROR - Top 5 Popular Movies
# ==============================
print(" TOP 5 POPULAR MOVIES - HORROR")
print("=" * 55)
display(get_top5_by_genre('Horror'))

 TOP 5 POPULAR MOVIES - HORROR


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,185,Psycho (1960),239,4.10,4.0593
2,183,Alien (1979),291,4.03,4.0025
3,208,Young Frankenstein (1974),200,3.94,3.9036
4,200,"Shining, The (1980)",206,3.83,3.7906
5,443,"Birds, The (1963)",162,3.81,3.7660


In [23]:
# ================================
# MUSICAL - Top 5 Popular Movies
# ================================
print("TOP 5 POPULAR MOVIES - MUSICAL")
print("=" * 55)
display(get_top5_by_genre('Musical'))

TOP 5 POPULAR MOVIES - MUSICAL


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,132,"Wizard of Oz, The (1939)",246,4.08,4.0381
2,705,Singin' in the Rain (1952),137,3.99,3.9303
3,209,This Is Spinal Tap (1984),191,3.91,3.8645
4,186,"Blues Brothers, The (1980)",251,3.84,3.8075
5,95,Aladdin (1992),219,3.81,3.7806


In [24]:
# ================================
# MYSTERY - Top 5 Popular Movies
# ================================
print(" TOP 5 POPULAR MOVIES - MYSTERY")
print("=" * 55)
display(get_top5_by_genre('Mystery'))

 TOP 5 POPULAR MOVIES - MYSTERY


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,603,Rear Window (1954),209,4.39,4.3277
2,479,Vertigo (1958),179,4.25,4.1892
3,513,"Third Man, The (1949)",72,4.33,4.1800
4,484,"Maltese Falcon, The (1941)",138,4.21,4.1335
5,302,L.A. Confidential (1997),297,4.16,4.1263


In [25]:
# ================================
# ROMANCE - Top 5 Popular Movies
# ================================
print(" TOP 5 POPULAR MOVIES - ROMANCE")
print("=" * 55)
display(get_top5_by_genre('Romance'))

 TOP 5 POPULAR MOVIES - ROMANCE


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,483,Casablanca (1942),243,4.46,4.4022
2,50,Star Wars (1977),583,4.36,4.3369
3,313,Titanic (1997),350,4.25,4.2132
4,172,"Empire Strikes Back, The (1980)",367,4.20,4.1744
5,173,"Princess Bride, The (1987)",324,4.17,4.1400


In [26]:
# ==============================
# SCI-FI - Top 5 Popular Movies
# ==============================
print(" TOP 5 POPULAR MOVIES - SCI-FI")
print("=" * 55)
display(get_top5_by_genre('Sci-Fi'))

 TOP 5 POPULAR MOVIES - SCI-FI


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,50,Star Wars (1977),583,4.36,4.3369
2,474,Dr. Strangelove or: How I Learned to Stop Worr...,194,4.25,4.1949
3,172,"Empire Strikes Back, The (1980)",367,4.20,4.1744
4,89,Blade Runner (1982),275,4.14,4.1009
5,183,Alien (1979),291,4.03,4.0025


In [27]:
# =================================
# THRILLER - Top 5 Popular Movies
# =================================
print("TOP 5 POPULAR MOVIES - THRILLER")
print("=" * 55)
display(get_top5_by_genre('Thriller'))

TOP 5 POPULAR MOVIES - THRILLER


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,408,"Close Shave, A (1995)",112,4.49,4.3751
2,12,"Usual Suspects, The (1995)",267,4.39,4.3385
3,603,Rear Window (1954),209,4.39,4.3277
4,98,"Silence of the Lambs, The (1991)",390,4.29,4.2594
5,480,North by Northwest (1959),179,4.28,4.2210


In [28]:
# ============================
# WAR - Top 5 Popular Movies
# ============================
print(" TOP 5 POPULAR MOVIES - WAR")
print("=" * 55)
display(get_top5_by_genre('War'))

 TOP 5 POPULAR MOVIES - WAR


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,318,Schindler's List (1993),298,4.47,4.4213
2,483,Casablanca (1942),243,4.46,4.4022
3,50,Star Wars (1977),583,4.36,4.3369
4,474,Dr. Strangelove or: How I Learned to Stop Worr...,194,4.25,4.1949
5,172,"Empire Strikes Back, The (1980)",367,4.20,4.1744


In [29]:
# =================================
# WESTERN - Top 5 Popular Movies
# =================================
print(" TOP 5 POPULAR MOVIES - WESTERN")
print("=" * 55)
display(get_top5_by_genre('Western'))

 TOP 5 POPULAR MOVIES - WESTERN


,Movie ID,Movie Title,No. of Ratings,Avg Rating,Popularity Score
Rank,,,,,
1,661,High Noon (1952),88,4.10,3.9976
2,435,Butch Cassidy and the Sundance Kid (1969),216,3.95,3.9104
3,510,"Magnificent Seven, The (1954)",121,3.94,3.8760
4,589,"Wild Bunch, The (1969)",43,4.02,3.8445
5,203,Unforgiven (1992),182,3.87,3.8269
